# 04 — Collaborative Filtering Model Training (Sprout-Style)

Train a denoising autoencoder on user rating patterns:
- **Input**: Sparse user rating vector (watched flag + normalized rating per anime)
- **Encoder**: Projects to 256-dim bottleneck with noise injection
- **Decoder**: Two heads — watch prediction (BCE) + rating prediction (Huber)
- **Loss**: Uncertainty-weighted combination (learnable per-head weights)

**Hardware**: T4 GPU (~16GB VRAM)

**Requires**: `02_preprocessing.ipynb` (needs `cf_ratings.npz` and `cf_anime_index.json`).

In [ ]:
!pip install -q torch scipy tqdm

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy import sparse
from pathlib import Path

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load CF data
cf_matrix = sparse.load_npz(DATA_DIR / "cf_ratings.npz")

with open(DATA_DIR / "cf_anime_index.json", "r") as f:
    anime_index = json.load(f)

n_users, n_anime = cf_matrix.shape
print(f"Rating matrix: {n_users:,} users × {n_anime:,} anime")
print(f"Non-zero ratings: {cf_matrix.nnz:,}")
print(f"Density: {cf_matrix.nnz / (n_users * n_anime):.4%}")

## Model Architecture

Following Sprout's design:
- Input: `anime_count * 2` (one slot for watched flag, one for normalized rating)
- Encoder: Dense → 1024 (Swish) → 256 bottleneck + Gaussian noise
- Decoder watch head: 256 → 1024 (Swish) → anime_count → Sigmoid
- Decoder rating head: 256 → 1024 (Swish) → anime_count
- Learnable uncertainty parameters for loss weighting

In [ ]:
class AnimeCFAutoencoder(nn.Module):
    def __init__(self, n_anime, bottleneck_dim=256, hidden_dim=1024, noise_std=0.1):
        super().__init__()
        self.n_anime = n_anime
        self.noise_std = noise_std
        input_dim = n_anime * 2  # watched flags + normalized ratings

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),  # Swish activation
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, bottleneck_dim)
        )

        # Decoder — watch prediction head
        self.watch_decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_anime)
        )

        # Decoder — rating prediction head
        self.rating_decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_anime)
        )

        # Learnable uncertainty parameters (Sprout-style)
        self.log_var_watch = nn.Parameter(torch.zeros(1))
        self.log_var_rating = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # Encode
        z = self.encoder(x)

        # Add noise during training (denoising autoencoder)
        if self.training:
            z = z + torch.randn_like(z) * self.noise_std

        # Decode
        watch_logits = self.watch_decoder(z)
        rating_pred = self.rating_decoder(z)

        return watch_logits, rating_pred

    def get_loss(self, watch_logits, rating_pred, watch_target, rating_target, rating_mask):
        """Uncertainty-weighted multi-task loss."""
        # Watch loss (BCE)
        watch_loss = F.binary_cross_entropy_with_logits(watch_logits, watch_target)

        # Rating loss (Huber, only for watched anime)
        if rating_mask.sum() > 0:
            rating_loss = F.huber_loss(
                rating_pred[rating_mask],
                rating_target[rating_mask],
                delta=1.0
            )
        else:
            rating_loss = torch.tensor(0.0, device=watch_logits.device)

        # Uncertainty weighting: L = (1/2σ²) * loss + log(σ)
        precision_watch = torch.exp(-self.log_var_watch)
        precision_rating = torch.exp(-self.log_var_rating)

        total_loss = (
            0.5 * precision_watch * watch_loss + 0.5 * self.log_var_watch +
            0.5 * precision_rating * rating_loss + 0.5 * self.log_var_rating
        )

        return total_loss, watch_loss.item(), rating_loss.item()


model = AnimeCFAutoencoder(n_anime).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Input dim: {n_anime * 2:,}, Bottleneck: 256, Output: {n_anime:,} per head")

In [ ]:
# Import from a real module so Windows multiprocessing workers can resolve the class.
import sys
from pathlib import Path
if not Path("training_datasets.py").exists() and Path("notebooks/training_datasets.py").exists():
    sys.path.append(str(Path("notebooks").resolve()))
from training_datasets import CFDataset

# Create dataset and split
full_dataset = CFDataset(cf_matrix)

train_size = int(0.95 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Train: {len(train_dataset):,}, Val: {len(val_dataset):,}")

In [ ]:
# Hyperparameters
BATCH_SIZE = 256
EPOCHS = 80
LR = 1e-3

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, verbose=True)

print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LR}")
print(f"Steps per epoch: {len(train_loader):,}")

In [ ]:
# Training loop
best_val_loss = float("inf")
cf_output_dir = MODEL_DIR / "anime_cf"
cf_output_dir.mkdir(exist_ok=True)

print("Starting CF training...")
for epoch in tqdm(range(EPOCHS), desc="CF epochs"):
    # Train
    model.train()
    train_losses = []
    train_watch_losses = []
    train_rating_losses = []

    for input_vec, watch_target, rating_target, rating_mask in tqdm(train_loader, desc=f"Train epoch {epoch+1}", leave=False):
        input_vec = input_vec.to(device)
        watch_target = watch_target.to(device)
        rating_target = rating_target.to(device)
        rating_mask = rating_mask.to(device)

        optimizer.zero_grad()
        watch_logits, rating_pred = model(input_vec)
        loss, w_loss, r_loss = model.get_loss(
            watch_logits, rating_pred, watch_target, rating_target, rating_mask
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())
        train_watch_losses.append(w_loss)
        train_rating_losses.append(r_loss)

    # Validate
    model.eval()
    val_losses = []
    with torch.no_grad():
        for input_vec, watch_target, rating_target, rating_mask in tqdm(val_loader, desc=f"Val epoch {epoch+1}", leave=False):
            input_vec = input_vec.to(device)
            watch_target = watch_target.to(device)
            rating_target = rating_target.to(device)
            rating_mask = rating_mask.to(device)

            watch_logits, rating_pred = model(input_vec)
            loss, _, _ = model.get_loss(
                watch_logits, rating_pred, watch_target, rating_target, rating_mask
            )
            val_losses.append(loss.item())

    avg_train = np.mean(train_losses)
    avg_val = np.mean(val_losses)
    scheduler.step(avg_val)

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            "model_state_dict": model.state_dict(),
            "n_anime": n_anime,
            "epoch": epoch
        }, cf_output_dir / "best_model.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Train: {avg_train:.4f} (watch={np.mean(train_watch_losses):.4f}, rating={np.mean(train_rating_losses):.4f}) | "
            f"Val: {avg_val:.4f} | "
            f"σ_watch={torch.exp(model.log_var_watch/2).item():.3f}, σ_rating={torch.exp(model.log_var_rating/2).item():.3f}"
        )

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {cf_output_dir / 'best_model.pt'}")

## Quick Validation

Test the model by predicting ratings for a few users.

In [ ]:
# Load best model
checkpoint = torch.load(cf_output_dir / "best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Build anime ID lookup
idx_to_info = {}
for entry in anime_index:
    idx_to_info[entry["idx"]] = {
        "mal_id": entry["mal_id"],
        "anilist_id": entry.get("anilist_id")
    }

# Load anime titles from Kaggle for display
KAGGLE_INPUT = Path("/kaggle/input/anime-recommendation-database-2020")
if not KAGGLE_INPUT.exists():
    KAGGLE_INPUT = DATA_DIR / "kaggle"
anime_df = pd.read_csv(KAGGLE_INPUT / "anime.csv")
mal_id_to_name = dict(zip(anime_df["MAL_ID"], anime_df["Name"]))

In [ ]:
import pandas as pd

# Pick a few random users and show their top predictions vs actual ratings
test_user_indices = np.random.choice(n_users, 3, replace=False)

for user_idx in test_user_indices:
    row = cf_matrix.getrow(user_idx).toarray().flatten()
    watched = (row != 0).astype(np.float32)
    ratings = row.astype(np.float32)

    # Full input (no dropout for inference)
    input_vec = np.concatenate([watched, ratings])
    input_tensor = torch.FloatTensor(input_vec).unsqueeze(0).to(device)

    with torch.no_grad():
        watch_logits, rating_pred = model(input_tensor)

    watch_probs = torch.sigmoid(watch_logits).cpu().numpy().flatten()
    pred_ratings = rating_pred.cpu().numpy().flatten()

    # Get top predictions for unwatched anime
    unwatched_mask = watched == 0
    scores = watch_probs * (pred_ratings + 5)  # Denormalize roughly
    scores[~unwatched_mask.astype(bool)] = -1  # Exclude watched

    top_indices = np.argsort(scores)[::-1][:10]

    print(f"\n{'='*60}")
    print(f"User {user_idx} | Watched: {int(watched.sum())} anime")
    print(f"{'='*60}")

    # Show some of their actual watched anime
    watched_indices = np.where(watched > 0)[0]
    print("\nSample watched (top rated):")
    watched_with_scores = [(i, ratings[i]) for i in watched_indices]
    watched_with_scores.sort(key=lambda x: x[1], reverse=True)
    for idx, score in watched_with_scores[:5]:
        info = idx_to_info.get(idx, {})
        name = mal_id_to_name.get(info.get("mal_id"), f"MAL#{info.get('mal_id', '?')}")
        print(f"  {name}: {score:+.1f} (raw)")

    print("\nTop 10 predictions:")
    for rank, idx in enumerate(top_indices, 1):
        info = idx_to_info.get(idx, {})
        name = mal_id_to_name.get(info.get("mal_id"), f"MAL#{info.get('mal_id', '?')}")
        print(f"  {rank}. {name} (watch_prob={watch_probs[idx]:.3f}, pred_rating={pred_ratings[idx]:+.2f})")

In [ ]:
# Save anime index mapping for the sidecar
print(f"\n✓ CF model saved to: {cf_output_dir / 'best_model.pt'}")
print(f"  Architecture: {n_anime*2} → 1024 → 256 → 1024 → {n_anime} (×2 heads)")
model_size = (cf_output_dir / 'best_model.pt').stat().st_size / 1e6
print(f"  Checkpoint size: {model_size:.1f} MB")
print(f"\nProceed to 05_export.ipynb")